# 07 Gold Layer and Data Quality

## Purpose
Create analysis-ready Gold datasets, run cross-table data-quality checks, and prepare a clean handoff to Phase 8.

Historical EEA data and current Open-Meteo context remain methodically separated. Spark and Kafka infrastructure are checked visibly, but this Gold-layer notebook does not claim shared FH cluster storage unless an explicit Spark write/readback probe succeeds.

## Inputs

- `data/silver/city_reference.parquet`
- `data/silver/city_metadata.parquet`
- `data/silver/eea_city_daily.parquet`
- Preferred live input: `data/silver/open_meteo_city_hourly/` from notebook `06`
- Local functional fallback: `data/bronze/open_meteo_raw/open_meteo_air_quality_events.jsonl` from notebook `05`

## Outputs

- `data/gold/city_air_quality_daily_summary.parquet`
- `data/gold/pollutant_ranking_by_city.parquet`
- `data/gold/city_context_air_quality.parquet`
- `data/gold/live_air_quality_latest.parquet`
- `data/gold/data_quality_summary.parquet`

## Technologies used
Python, pandas, pyarrow, optional PySpark storage probe, socket connectivity checks, Parquet.

This follows the course-reference patterns for file loading, joins, `groupby`, aggregation, descriptive checks, and Parquet write/readback. Spark Structured Streaming remains implemented in notebook `06`.

## Configuration

The Gold layer writes driver-local Parquet files for reproducibility. The notebook checks the configured Kafka broker and Spark master via TCP. Set `RUN_PHASE7_SPARK_STORAGE_PROBE=true` only when Spark is installed and you want to verify that the selected Spark master can write and read `DATA_DIR`.

If Phase-6 Silver streaming Parquet is absent locally, the notebook reconstructs the live snapshot from validated Phase-5 JSONL events. This is an explicit mechanics fallback, not FH Kafka-to-Spark evidence.

### Resolve the repository root

The notebook loads `.env` from the repository and supports execution from both the root folder and the `notebooks/` folder.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime, timezone
import json
import os
import shutil
import socket
import pandas as pd

_cwd = Path.cwd().resolve()
_candidate_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
load_dotenv(_candidate_root / ".env")
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", _candidate_root)).resolve()


### Define reusable project-relative paths

This helper resolves relative data settings below the repository root while preserving absolute shared-storage paths when configured.


In [ ]:
def project_path(env_name: str, default: str) -> Path:
    path = Path(os.getenv(env_name, default))
    return path if path.is_absolute() else PROJECT_ROOT / path


### Declare Silver and Gold datasets

All input and output contracts are visible in one place. This makes the Gold-layer boundary easy to review.


In [ ]:
DATA_DIR = project_path("DATA_DIR", "data")
CHECKPOINT_DIR = project_path("CHECKPOINT_DIR", "data/checkpoints")
GOLD_DIR = DATA_DIR / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"
CITY_METADATA_PATH = DATA_DIR / "silver" / "city_metadata.parquet"
EEA_DAILY_PATH = DATA_DIR / "silver" / "eea_city_daily.parquet"
OPEN_METEO_SILVER_PATH = DATA_DIR / "silver" / "open_meteo_city_hourly"
PHASE5_EVENTS_PATH = DATA_DIR / "bronze" / "open_meteo_raw" / "open_meteo_air_quality_events.jsonl"

DAILY_SUMMARY_PATH = GOLD_DIR / "city_air_quality_daily_summary.parquet"
RANKING_PATH = GOLD_DIR / "pollutant_ranking_by_city.parquet"
CONTEXT_PATH = GOLD_DIR / "city_context_air_quality.parquet"
LIVE_LATEST_PATH = GOLD_DIR / "live_air_quality_latest.parquet"
QUALITY_PATH = GOLD_DIR / "data_quality_summary.parquet"


### Load infrastructure flags

Phase 7 reads the configured Spark master, Kafka broker, topic and optional storage-probe flag without starting a producer.


In [ ]:
SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "local[*]")
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "<kafka-host>:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC_AIR_QUALITY_LIVE", "LIVE-bdeng_gXX_air_quality_live")
RUN_OPEN_METEO_KAFKA_PRODUCER = os.getenv("RUN_OPEN_METEO_KAFKA_PRODUCER", "false").lower() == "true"
RUN_PHASE7_SPARK_STORAGE_PROBE = os.getenv("RUN_PHASE7_SPARK_STORAGE_PROBE", "false").lower() == "true"


### Define a lightweight TCP connectivity check

The probe reports placeholder settings and unreachable endpoints as data instead of hiding infrastructure limitations.


In [ ]:
def tcp_check(endpoint: str, timeout_seconds: float = 2.0) -> dict:
    if "<" in endpoint or ":" not in endpoint:
        return {"endpoint": endpoint, "reachable": False, "reason": "placeholder or malformed endpoint"}
    host, port = endpoint.rsplit(":", 1)
    try:
        with socket.create_connection((host, int(port)), timeout=timeout_seconds):
            return {"endpoint": endpoint, "reachable": True, "reason": None}
    except Exception as exc:
        return {"endpoint": endpoint, "reachable": False, "reason": str(exc)}


### Check Kafka and Spark endpoints

Local Spark mode is considered available locally. Remote endpoints receive an explicit TCP connectivity result.


In [ ]:
spark_endpoint = SPARK_MASTER_URL.removeprefix("spark://") if SPARK_MASTER_URL.startswith("spark://") else SPARK_MASTER_URL
infrastructure_status = {
    "kafka": tcp_check(KAFKA_BOOTSTRAP_SERVERS),
    "spark_master": tcp_check(spark_endpoint) if SPARK_MASTER_URL.startswith("spark://") else {
        "endpoint": SPARK_MASTER_URL, "reachable": True, "reason": "local Spark mode"
    },
}
print({
    "project_root": str(PROJECT_ROOT),
    "spark_master_url": SPARK_MASTER_URL,
    "kafka_bootstrap_servers": KAFKA_BOOTSTRAP_SERVERS,
    "kafka_topic": KAFKA_TOPIC,
    "run_open_meteo_kafka_producer": RUN_OPEN_METEO_KAFKA_PRODUCER,
    "run_phase7_spark_storage_probe": RUN_PHASE7_SPARK_STORAGE_PROBE,
    "infrastructure_status": infrastructure_status,
})


## Implementation

### Readiness check for phases 0 to 6

Notebook `07` verifies repository structure, dependency files, schemas, join keys, provenance, and optional cluster storage. Missing Phase-6 Silver live data selects an explicit local reconstruction path instead of silently pretending that Spark consumed Kafka.

#### Verify notebook order and required files

Phase 7 first checks that notebooks `00` to `07` and mandatory Silver inputs exist and that no accidental `notebooks/data/` output folder was created.


In [ ]:
required_notebooks = [PROJECT_ROOT / "notebooks" / f"{index:02d}_{name}.ipynb" for index, name in [
    (0, "project_scope_and_requirements"),
    (1, "source_spike_and_cluster_check"),
    (2, "city_reference_model"),
    (3, "eea_batch_ingestion"),
    (4, "wikipedia_web_scraping"),
    (5, "open_meteo_api_and_kafka_producer"),
    (6, "spark_structured_streaming_kafka_to_parquet"),
    (7, "gold_layer_and_data_quality"),
]]
for path in required_notebooks:
    assert path.exists(), f"Missing notebook: {path}"
assert not (PROJECT_ROOT / "notebooks" / "data").exists(), "Wrong output folder exists: notebooks/data"
assert not RUN_OPEN_METEO_KAFKA_PRODUCER, "Disable RUN_OPEN_METEO_KAFKA_PRODUCER for Phase 7"
for path in [CITY_REFERENCE_PATH, CITY_METADATA_PATH, EEA_DAILY_PATH]:
    assert path.exists(), f"Missing required Silver input: {path}"


#### Load Silver datasets

The Gold layer reads the shared city model, scraped context and historical EEA daily aggregates.


In [ ]:
city_reference_df = pd.read_parquet(CITY_REFERENCE_PATH)
city_metadata_df = pd.read_parquet(CITY_METADATA_PATH)
eea_daily_df = pd.read_parquet(EEA_DAILY_PATH)


#### Validate Silver contracts and joins

Required columns, city uniqueness, pollutant scope and joinability are asserted before any Gold data is written.


In [ ]:
required_contracts = {
    "city_reference": (city_reference_df, {"city_id", "city_name", "country_code", "latitude", "longitude"}),
    "city_metadata": (city_metadata_df, {"city_id", "population", "area_km2", "population_density", "parse_status"}),
    "eea_daily": (eea_daily_df, {"city_id", "date", "pollutant", "mean_value", "min_value", "max_value", "observation_count", "source", "data_status"}),
}
for name, (frame, columns) in required_contracts.items():
    missing = columns - set(frame.columns)
    assert not missing, f"{name} missing required columns: {sorted(missing)}"

assert len(city_reference_df) >= 8 and city_reference_df["city_id"].is_unique
assert city_metadata_df["city_id"].is_unique
assert set(eea_daily_df["city_id"]).issubset(set(city_reference_df["city_id"]))
assert set(city_metadata_df["city_id"]).issubset(set(city_reference_df["city_id"]))
assert set(eea_daily_df["pollutant"]).issubset({"pm2_5", "pm10", "no2"})


#### Optionally test Spark-worker storage

The probe is disabled by default. When enabled on FH JupyterHub, it writes and rereads a tiny Parquet dataset using the configured Spark master.


In [ ]:
spark_storage_status = {"probe_requested": RUN_PHASE7_SPARK_STORAGE_PROBE, "passed": False, "reason": "not requested"}
if RUN_PHASE7_SPARK_STORAGE_PROBE:
    probe_path = DATA_DIR / "_phase7_spark_storage_probe"
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.appName("phase7-storage-probe").master(SPARK_MASTER_URL).getOrCreate()
        spark.createDataFrame([(1, "ok")], ["id", "status"]).write.mode("overwrite").parquet(str(probe_path))
        assert spark.read.parquet(str(probe_path)).count() == 1
        spark_storage_status = {"probe_requested": True, "passed": True, "reason": None}
    except Exception as exc:
        spark_storage_status = {"probe_requested": True, "passed": False, "reason": str(exc)}
    finally:
        try:
            spark.stop()
        except Exception:
            pass
        shutil.rmtree(probe_path, ignore_errors=True)


#### Report readiness and provenance

The EEA `data_status` marker determines whether final empirical claims are allowed later.


In [ ]:
eea_sample_fallback_used = set(eea_daily_df["data_status"]) != {"real_eea_file"}
print({
    "city_reference_rows": len(city_reference_df),
    "city_metadata_rows": len(city_metadata_df),
    "eea_daily_rows": len(eea_daily_df),
    "eea_data_status": sorted(eea_daily_df["data_status"].unique()),
    "eea_sample_fallback_used": eea_sample_fallback_used,
    "spark_storage_status": spark_storage_status,
})


### Build historical Gold tables

The historical Gold tables use EEA daily aggregates only. Open-Meteo live values are not mixed into historical rankings.

#### Join historical EEA values with city context

The historical branch uses EEA daily rows only and adds display fields plus Wikipedia context.


In [ ]:
eea_enriched_df = (
    eea_daily_df
    .merge(city_reference_df[["city_id", "city_name", "country_code", "latitude", "longitude"]], on="city_id", how="left", validate="many_to_one")
    .merge(city_metadata_df[["city_id", "population", "area_km2", "population_density", "parse_status"]], on="city_id", how="left", validate="many_to_one")
)
assert eea_enriched_df["city_name"].notna().all(), "EEA rows without city reference join found"
eea_enriched_df["dataset_context"] = "eea_historical"


#### Create the historical daily Gold table

This table keeps one reviewable row per city, date and pollutant and renames measures for the visualization boundary.


In [ ]:
city_air_quality_daily_summary_df = eea_enriched_df[[
    "city_id", "city_name", "country_code", "date", "pollutant", "unit",
    "mean_value", "min_value", "max_value", "observation_count", "source",
    "data_status", "dataset_context",
]].rename(columns={"mean_value": "avg_value", "observation_count": "measurement_count"})


#### Calculate pollutant-specific city rankings

Mean values, ranges, observation counts and a dense rank prepare the primary Phase-8 comparison.


In [ ]:
pollutant_ranking_by_city_df = (
    city_air_quality_daily_summary_df
    .groupby(["city_id", "city_name", "country_code", "pollutant", "unit", "data_status", "dataset_context"], as_index=False)
    .agg(
        mean_pollutant_value=("avg_value", "mean"),
        min_pollutant_value=("min_value", "min"),
        max_pollutant_value=("max_value", "max"),
        daily_observation_count=("date", "count"),
        measurement_count=("measurement_count", "sum"),
    )
)
pollutant_ranking_by_city_df["pollutant_rank"] = (
    pollutant_ranking_by_city_df.groupby("pollutant")["mean_pollutant_value"]
    .rank(method="dense", ascending=False).astype(int)
)


#### Add Wikipedia context to rankings

Population, area and density support exploratory context plots without implying causality.


In [ ]:
city_context_air_quality_df = pollutant_ranking_by_city_df.merge(
    city_metadata_df[["city_id", "population", "area_km2", "population_density", "parse_status"]],
    on="city_id", how="left", validate="many_to_one",
)


#### Write and read back historical Gold Parquets

Every historical output is persisted and immediately reread so Phase 8 receives verified files.


In [ ]:
for frame, path in [
    (city_air_quality_daily_summary_df, DAILY_SUMMARY_PATH),
    (pollutant_ranking_by_city_df, RANKING_PATH),
    (city_context_air_quality_df, CONTEXT_PATH),
]:
    frame.to_parquet(path, index=False)
    assert len(pd.read_parquet(path)) == len(frame), f"Parquet readback mismatch: {path}"

print({
    "daily_summary_rows": len(city_air_quality_daily_summary_df),
    "ranking_rows": len(pollutant_ranking_by_city_df),
    "context_rows": len(city_context_air_quality_df),
})
pollutant_ranking_by_city_df.sort_values(["pollutant", "pollutant_rank"]).head(12)


### Finalize the separate live snapshot

Phase-6 Silver Parquet is preferred. For local reproducibility, Phase-5 JSONL events can reconstruct the latest snapshot. The output records `live_input_mode` so a fallback cannot be mistaken for Kafka-to-Spark evidence.

#### Define the Parquet reader

The tiny wrapper keeps the preferred Phase-6 Silver path explicit.


In [ ]:
def read_parquet_dataset(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path)


#### Select Phase-6 Silver or the local JSONL fallback

Phase-6 Parquet is preferred. The Phase-5 reconstruction path is allowed only as an explicitly labeled local mechanics fallback.


In [ ]:
if OPEN_METEO_SILVER_PATH.exists():
    open_meteo_hourly_df = read_parquet_dataset(OPEN_METEO_SILVER_PATH)
    live_input_mode = "phase6_spark_stream_silver"
else:
    assert PHASE5_EVENTS_PATH.exists(), "Missing live fallback JSONL. Run notebook 05 before notebook 07."
    events = [json.loads(line) for line in PHASE5_EVENTS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    open_meteo_hourly_df = pd.DataFrame(events)
    live_input_mode = "phase5_jsonl_mock_reconstruction"


#### Validate and deduplicate live events

The event contract is checked and timestamp columns are normalized before the newest event per city is selected.


In [ ]:
required_live_columns = {"event_id", "city_id", "event_time_utc", "ingestion_time_utc", "data_status", "pm2_5", "pm10", "no2"}
missing_live = required_live_columns - set(open_meteo_hourly_df.columns)
assert not missing_live, f"Live input missing required columns: {sorted(missing_live)}"
open_meteo_hourly_df["event_time_ts"] = pd.to_datetime(open_meteo_hourly_df["event_time_utc"], utc=True, errors="coerce")
open_meteo_hourly_df["ingestion_time_ts"] = pd.to_datetime(open_meteo_hourly_df["ingestion_time_utc"], utc=True, errors="coerce")
assert open_meteo_hourly_df[["event_time_ts", "ingestion_time_ts"]].notna().all().all()
open_meteo_hourly_df = open_meteo_hourly_df.drop_duplicates("event_id")


#### Write the separate live snapshot

The snapshot is joined with city context, marked `open_meteo_live`, persisted and read back independently from historical tables.


In [ ]:
live_air_quality_latest_df = (
    open_meteo_hourly_df
    .sort_values(["city_id", "event_time_ts", "ingestion_time_ts"], ascending=[True, False, False])
    .drop_duplicates("city_id", keep="first")
    .merge(city_reference_df[["city_id", "city_name", "country_code", "latitude", "longitude"]], on="city_id", how="left", validate="one_to_one")
    .merge(city_metadata_df[["city_id", "population", "area_km2", "population_density", "parse_status"]], on="city_id", how="left", validate="one_to_one")
)
live_air_quality_latest_df["dataset_context"] = "open_meteo_live"
live_air_quality_latest_df["live_input_mode"] = live_input_mode
live_air_quality_latest_df.to_parquet(LIVE_LATEST_PATH, index=False)
live_readback_df = pd.read_parquet(LIVE_LATEST_PATH)
assert live_readback_df["city_id"].is_unique
assert set(live_readback_df["dataset_context"]) == {"open_meteo_live"}
print({"live_input_mode": live_input_mode, "live_latest_rows": len(live_readback_df)})
live_readback_df[["city_id", "city_name", "event_time_ts", "data_status", "live_input_mode", "pm2_5", "pm10", "no2"]]


## Validation / Quality Checks

The quality report profiles every Gold dataset and records key integrity checks. It keeps historical and live provenance visible so Phase 8 can tell a coherent but appropriately limited story.

#### Define dataset-level profiling

The profile summarizes rows, columns, duplicate keys, missing values, city coverage and provenance for each Gold dataset.


In [ ]:
def dataset_profile(name: str, frame: pd.DataFrame, key_columns: list[str]) -> dict:
    duplicate_keys = int(frame.duplicated(key_columns).sum())
    return {
        "dataset": name,
        "row_count": len(frame),
        "column_count": len(frame.columns),
        "duplicate_key_count": duplicate_keys,
        "missing_value_count": int(frame.isna().sum().sum()),
        "city_count": int(frame["city_id"].nunique()) if "city_id" in frame else None,
        "dataset_contexts": ",".join(sorted(frame["dataset_context"].dropna().astype(str).unique())) if "dataset_context" in frame else None,
        "data_statuses": ",".join(sorted(frame["data_status"].dropna().astype(str).unique())) if "data_status" in frame else None,
        "profiled_at_utc": datetime.now(timezone.utc).isoformat(),
    }


#### Profile all Gold datasets

Historical tables and the separate live snapshot are evaluated with keys appropriate to their grain.


In [ ]:
profiles = [
    dataset_profile("city_air_quality_daily_summary", city_air_quality_daily_summary_df, ["city_id", "date", "pollutant"]),
    dataset_profile("pollutant_ranking_by_city", pollutant_ranking_by_city_df, ["city_id", "pollutant"]),
    dataset_profile("city_context_air_quality", city_context_air_quality_df, ["city_id", "pollutant"]),
    dataset_profile("live_air_quality_latest", live_readback_df, ["city_id"]),
]
data_quality_summary_df = pd.DataFrame(profiles)


#### Calculate pollutant plausibility findings

Historical and live values are checked against the documented pollutant ranges independently.


In [ ]:
plausibility_limits = {"pm2_5": 1000, "pm10": 2000, "no2": 1000}
historical_invalid = sum(
    int(((city_air_quality_daily_summary_df["pollutant"] == pollutant) &
         ~city_air_quality_daily_summary_df["avg_value"].between(0, maximum)).sum())
    for pollutant, maximum in plausibility_limits.items()
)
live_invalid = sum(
    int((~live_readback_df[pollutant].between(0, maximum) & live_readback_df[pollutant].notna()).sum())
    for pollutant, maximum in plausibility_limits.items()
)


#### Enforce Gold-layer invariants

Assertions prevent duplicates, context mixing, implausible values and accidental writes below `notebooks/data/`.


In [ ]:
assert set(city_air_quality_daily_summary_df["dataset_context"]) == {"eea_historical"}
assert set(pollutant_ranking_by_city_df["dataset_context"]) == {"eea_historical"}
assert set(city_context_air_quality_df["dataset_context"]) == {"eea_historical"}
assert historical_invalid == 0, f"Invalid historical pollutant rows: {historical_invalid}"
assert live_invalid == 0, f"Invalid live pollutant rows: {live_invalid}"
assert not data_quality_summary_df["duplicate_key_count"].any(), data_quality_summary_df
assert not (PROJECT_ROOT / "notebooks" / "data").exists()


#### Persist the quality report

Infrastructure checks and fallback modes become columns in the Parquet report so Phase 8 and reviewers can inspect the actual run context.


In [ ]:
data_quality_summary_df["eea_sample_fallback_used"] = eea_sample_fallback_used
data_quality_summary_df["live_input_mode"] = live_input_mode
data_quality_summary_df["kafka_tcp_reachable"] = infrastructure_status["kafka"]["reachable"]
data_quality_summary_df["spark_master_tcp_reachable"] = infrastructure_status["spark_master"]["reachable"]
data_quality_summary_df["spark_storage_probe_passed"] = spark_storage_status["passed"]
data_quality_summary_df.to_parquet(QUALITY_PATH, index=False)
quality_readback_df = pd.read_parquet(QUALITY_PATH)
quality_readback_df


### Gold readback and Phase-8 handoff

#### Read back all Gold outputs

The handoff begins by confirming that every expected Gold file exists, is readable and contains rows.


In [ ]:
gold_paths = {
    "city_air_quality_daily_summary": DAILY_SUMMARY_PATH,
    "pollutant_ranking_by_city": RANKING_PATH,
    "city_context_air_quality": CONTEXT_PATH,
    "live_air_quality_latest": LIVE_LATEST_PATH,
    "data_quality_summary": QUALITY_PATH,
}
gold_readback = {}
for name, path in gold_paths.items():
    assert path.exists(), f"Missing Gold output: {path}"
    frame = pd.read_parquet(path)
    assert len(frame) > 0, f"Empty Gold output: {path}"
    gold_readback[name] = {"rows": len(frame), "columns": list(frame.columns)}
print(gold_readback)


#### Evaluate Phase-8 readiness

The final dictionary states whether rankings, context plots and a separate live snapshot are available and whether real EEA data allows final analytical claims.


In [ ]:
phase8_readiness = {
    "historical_rankings_available": {"pollutant", "pollutant_rank", "mean_pollutant_value"}.issubset(pollutant_ranking_by_city_df.columns),
    "context_plot_available": {"population_density", "mean_pollutant_value"}.issubset(city_context_air_quality_df.columns),
    "live_snapshot_available": len(live_readback_df) > 0,
    "historical_and_live_separated": (
        set(city_context_air_quality_df["dataset_context"]) == {"eea_historical"}
        and set(live_readback_df["dataset_context"]) == {"open_meteo_live"}
    ),
    "final_analytical_claims_allowed": not eea_sample_fallback_used,
}
assert all(value for key, value in phase8_readiness.items() if key != "final_analytical_claims_allowed")
print({"phase8_readiness": phase8_readiness})
print("Phase 7 completed. Phase 8 may visualize Gold outputs only.")


## Results

Phase 7 creates five Gold datasets and a visible quality report. The final output explicitly reports whether real EEA data is available and whether the live snapshot came from Phase-6 Spark Silver Parquet or the local Phase-5 JSONL fallback.

If `final_analytical_claims_allowed=False`, Phase 8 may demonstrate visualization mechanics but must not present the controlled EEA sample as empirical evidence.

## Limitations

- Gold production is intentionally driver-local until shared FH Spark-worker storage is proven by a successful probe.
- A Phase-5 JSONL reconstruction of live data is a local mechanics fallback, not Spark-from-Kafka evidence.
- Controlled EEA sample data keeps the pipeline reproducible but cannot support final analytical conclusions.
- Wikipedia metadata is contextual and heuristic; correlations remain exploratory, not causal.

## Next step
Run notebook `08_analysis_visualization_and_storytelling.ipynb`. It must consume Gold outputs only and keep sample-based, live, and historical interpretations clearly labeled.